## Atividade Otimização 
### Part 1 - Introdução
 - PPGCAP - INPE 2026.3
 - Dicente: William Max
 - Docente: Fabricio G


### TC.1

Titulo: Applying a Dynamic Data Driven Genetci Algorithm to Improve Forest Fire Spread Prediction

a. Qual problema está sendo resolvido.

    O caso abordado no artigo em que se visa otmização é busca de melhoria na seleção das váriaveis/cenários de input para o sistema que simula a previsão de crescimento de incêncios.

b. Em qual contexto a otimização está sendo efetuada e qual técnica é utilizada.

    A técnica utilizada foi a de algoritmo genético junto ao paradigma  Dynamic
Data Driven Application Systems (DDDAS) Não entendi bem esse paradigma. Classiacemtne as predições de crecimento de incendios rodam a equação matemática de R e tem como entrada parâmetros com aclive, direção do vento e etc. Sendo assim, encontrar os cenários com os melhores conjuntos representativos torna a simulação melhor minimizando o erro entre o real e simulado. 
    and real fire progress to obtain the main wind characteristics for performing steering method through Calibration stage

c. Quais as variáveis de projeto/decisão do problema.

    Aclive (direção e inclinação), direção do vento e velocidade do vento e tempo considerando a inclinação e tipo de vegetação são considerados estáticos

d. Como uma solução candidata é representada na obtenção da solução ótima.

    A que tem menor erro com relação o caso real de expanção do evento

e. Qual o critério de avaliação da solução (i.e., função objetivo).

    Função objetivo é minimizar o erro 

f. Qual/quais os critérios de parada utilizados.

    5 gerações pois foi visto mpouca melhoria na minimizaçao de erros após 3 ou 4

g. Quais as restrições do problema de otimização e como elas são tratadas.




a. Problema: Reduzir a incerteza dos dados de entrada em simuladores de incêndio para que a previsão acompanhe o comportamento real do fogo.   

b. Contexto e Técnica: Otimização para calibração de simuladores em tempo real (paradigma DDDAS), utilizando como técnica um Algoritmo Genético Guiado por Dados (Dynamic Data Driven GA).   

c. Variáveis de decisão: Direção do vento, velocidade do vento, umidade do combustível vivo e umidade do combustível morto em três tempos distintos.   

d. Representação da solução: Cada solução candidata é representada como um "indivíduo" (um cenário específico) que carrega um vetor com os valores testados para as variáveis de decisão.   

e. Função objetivo: Minimizar o erro de propagação, medido pela diferença de semelhança entre o mapa de fogo simulado e o mapa de fogo real observado.   

f. Critério de parada: O limite máximo de 5 iterações (gerações), estabelecido porque a redução do erro se torna insignificante a partir da terceira ou quarta iteração.   

g. Restrições e Tratamento: As restrições do espaço de busca são tratadas por engenharia reversa. O sistema analisa a posição real do fogo e o declive topográfico para deduzir analiticamente os vetores do vento. Esses dados deduzidos são injetados nas operações de elitismo e mutação, restringindo o GA a buscar soluções apenas em zonas com viabilidade física comprovada.


Definir o método

Dizer o que é um simplex

Explicar operações

explicar comparações e procediemtnos

Work flow para implementação do algoritmo de Nealder-Mead
 
 - criar função que descobre a dimenção da função

 - criar função que cria os prontos do simplex (Talvez dando o centroid e ele criando os outro)

 - Criar funcção (acho que só ordenar a lista) ordena do pior para o melhor ponto do simplex tendo em vista suas umagens no ponto

 - Criar uma função para encontrar o centroid

 - função reflexão

 - função expanão
 
 - função contração 

Comparar com busca exaustiva

Imagem ilustrando o desenvolvimento do algoritmo



In [2]:
import numpy as np

In [3]:
first = np.array([0,0])

In [4]:
def define_simplex(kickoff, step):
    """Cria os vértices do simplex a partir de um ponto inicial e um step."""
    dim = len(kickoff)
    
    simplex = [kickoff]

    for i in range(dim):
        new_point = kickoff.copy()  
        new_point[i] += step        
        simplex.append(new_point)  
    
    return np.array(simplex)      

In [5]:
def centroid (goods_points):
    """Determina o centroid sem o pior elemento"""
    return np.mean(goods_points, axis=0)

In [6]:
def select_good_points(new_simplex, func):
    """
    Avalia a função nos pontos do simplex, ordena do melhor para o pior,
    e separa os pontos que usaremos para o centroide.
    """

    #passa os pontos na função e ordena os simplex de modo crecente conforme f(simplex)
    simplex_ordenado = sorted(new_simplex, key=func)
    
    goods_points = simplex_ordenado[:-1]
    
    worst_point = simplex_ordenado[-1]
    
    return simplex_ordenado, goods_points, worst_point

In [7]:
alpha = 1
yota = 2
rho = sigma =0.5 

def iteracao_nelder_mead(simplex, func):

    simplex_ordenado, goods_points, worst_point = select_good_points(simplex, func)

    centroide = centroid(goods_points)

    ponto_refletido = centroide + alpha*(centroide - worst_point)


    best_point = simplex_ordenado[0]

    ponto_refletido_value = func(ponto_refletido)
    best_point_value = func(simplex_ordenado[0])
    befor_wrost_value= func(simplex_ordenado[-2])
    worst_value = func(simplex_ordenado[-1])

    # 4. A Árvore de Decisão
    if ponto_refletido_value < best_point_value:
        # Entense-se que é um declive e o empurra para um lugar mais
        ponto_expandido = centroide + yota*(ponto_refletido - centroide)
        #Teste para validar o ponto exandido não estra em um aclive
        ponto_expandido_value = func(ponto_expandido)
        
        # Só aceita a expansão se ela for realmente melhor que a reflexão
        if ponto_expandido_value < ponto_refletido_value:
            simplex_ordenado[-1] = ponto_expandido
        else:
            simplex_ordenado[-1] = ponto_refletido


    elif ponto_refletido_value < befor_wrost_value:
    
        simplex_ordenado[-1] = ponto_refletido
        
    else:

        if ponto_refletido_value < worst_value: 
            ref = ponto_refletido
        else: 
            ref = worst_point
            
        ponto_retracao = centroide + rho * (ref - centroide)
        ponto_retracao_value = func(ponto_retracao)
        
        # Testa se a contração realmente ajudou
        if ponto_retracao_value < worst_value:
            simplex_ordenado[-1] = ponto_retracao
        else:
            #Encolhimento (Shrink)
            for i in range(1, len(simplex_ordenado)):
                ponto_encolhido = best_point + sigma * (simplex_ordenado[i] - best_point)
                simplex_ordenado[i] = ponto_encolhido
        
    return simplex_ordenado
        

In [8]:
def nelder_mead_otimizador(func, kickoff, step=1.0, tol=1e-6, max_iter=1000):
    """
    Wrapper principal que executa o Nelder-Mead.
    """
    # 1. Geração inicial (usando a sua primeira função)
    simplex = define_simplex(kickoff, step)
    
    iters = 0
    while iters < max_iter:
        # 2. Critério de Parada (Convergência)
        # Calculamos as altitudes de todos os pontos atuais.
        valores = np.array([func(p) for p in simplex])
        
        # Se o desvio padrão das altitudes for menor que a tolerância,
        # significa que os vértices encolheram tanto que estão praticamente 
        # no mesmo lugar (fundo do poço). Podemos parar!
        if np.std(valores) < tol:
            break
            
        # 3. Roda o motor (a sua função com a árvore de decisão)
        simplex = iteracao_nelder_mead(simplex, func)
        
        iters += 1

    # 4. Fim do laço. O simplex está no fundo do vale. 
    # Ordenamos uma última vez só para garantir que vamos retornar o melhor vértice.
    valores_finais = np.array([func(p) for p in simplex])
    indice_melhor = np.argsort(valores_finais)[0]
    
    melhor_ponto = simplex[indice_melhor]
    melhor_valor = valores_finais[indice_melhor]
    
    return melhor_ponto, melhor_valor, iters

In [9]:
# Função objetivo para teste
def funcao_esfera(ponto):
    x, y = ponto[0], ponto[1]
    return (x-1)**2 + (y+2)**2

# Chute inicial (bem longe da resposta)
kickoff = np.array([5.0, 3.0])

# Executando o SEU algoritmo
ponto_otimo, valor_otimo, iteracoes = nelder_mead_otimizador(funcao_esfera, kickoff, step=1.0)

print(f"Ponto encontrado: {ponto_otimo}")
print(f"Valor da função: {valor_otimo:.8f}")
print(f"Iterações gastas: {iteracoes}")

Ponto encontrado: [ 0.99914692 -2.00130682]
Valor da função: 0.00000244
Iterações gastas: 27


In [10]:
# ==========================================
# 1. FUNÇÕES AUXILIARES DO MOTOR
# ==========================================
def define_simplex(kickoff, step):
    dim = len(kickoff)
    simplex = [kickoff]
    for i in range(dim):
        new_point = kickoff.copy()
        new_point[i] += step
        simplex.append(new_point)
    return np.array(simplex)

def sort_and_select(simplex, func):
    valores = np.array([func(ponto) for ponto in simplex])
    indices_ordenados = np.argsort(valores)
    simplex_ordenado = simplex[indices_ordenados]
    
    goods_points = simplex_ordenado[:-1]
    worst_point = simplex_ordenado[-1]
    return simplex_ordenado, goods_points, worst_point

def get_centroid(goods_points):
    return np.mean(goods_points, axis=0)

# ==========================================
# 2. A ITERAÇÃO (A sua Árvore de Decisão)
# ==========================================
def iteracao_nelder_mead(simplex, func):
    alpha = 1.0
    gamma = 2.0  # yota
    rho = 0.5
    sigma = 0.5

    simplex_ordenado, goods_points, worst_point = sort_and_select(simplex, func)
    centroide = get_centroid(goods_points)

    # Reflexão
    ponto_refletido = centroide + alpha * (centroide - worst_point)
    
    # Valores para decisão
    best_point = simplex_ordenado[0]
    valor_refletido = func(ponto_refletido)
    valor_best = func(best_point)
    valor_second_worst = func(simplex_ordenado[-2])
    valor_worst = func(worst_point)

    # Árvore de Decisão
    if valor_refletido < valor_best:
        # EXPANSÃO
        ponto_expandido = centroide + gamma * (ponto_refletido - centroide)
        if func(ponto_expandido) < valor_refletido:
            simplex_ordenado[-1] = ponto_expandido
        else:
            simplex_ordenado[-1] = ponto_refletido

    elif valor_refletido < valor_second_worst:
        # ACEITAÇÃO
        simplex_ordenado[-1] = ponto_refletido
        
    else:
        # CONTRAÇÃO
        if valor_refletido < valor_worst: 
            ref = ponto_refletido
        else: 
            ref = worst_point
            
        ponto_retracao = centroide + rho * (ref - centroide)
        
        if func(ponto_retracao) < valor_worst:
            simplex_ordenado[-1] = ponto_retracao
        else:
            # ENCOLHIMENTO (Shrink)
            for i in range(1, len(simplex_ordenado)):
                simplex_ordenado[i] = best_point + sigma * (simplex_ordenado[i] - best_point)
                
    return simplex_ordenado

# ==========================================
# 3. O WRAPPER PRINCIPAL
# ==========================================
def nelder_mead_otimizador(func, kickoff, step=1.0, tol=1e-6, max_iter=1000):
    simplex = define_simplex(kickoff, step)
    
    iters = 0
    while iters < max_iter:
        valores = np.array([func(p) for p in simplex])
        
        # Critério de parada: se o desvio padrão for muito pequeno
        if np.std(valores) < tol:
            break
            
        simplex = iteracao_nelder_mead(simplex, func)
        iters += 1

    valores_finais = np.array([func(p) for p in simplex])
    indice_melhor = np.argsort(valores_finais)[0]
    
    return simplex[indice_melhor], valores_finais[indice_melhor], iters

# ==========================================
# 4. EXECUÇÃO DO TESTE
# ==========================================
def funcao_esfera(ponto):
    x, y = ponto[0], ponto[1]
    # O mínimo real desta função está em x=1, y=-2
    return (x - 1)**2 + (y + 2)**2

kickoff = np.array([5.0, 3.0])

ponto_otimo, valor_otimo, iteracoes = nelder_mead_otimizador(funcao_esfera, kickoff, step=1.0)

print(f"Mínimo real esperado: [1.0, -2.0]")
print(f"Ponto encontrado: [{ponto_otimo[0]:.4f}, {ponto_otimo[1]:.4f}]")
print(f"Valor da função: {valor_otimo:.8f}")
print(f"Iterações gastas: {iteracoes}")

Mínimo real esperado: [1.0, -2.0]
Ponto encontrado: [0.9991, -2.0013]
Valor da função: 0.00000244
Iterações gastas: 27
